In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
USE CATALOG `retail-sales-data-warehouse`;
USE SCHEMA gold;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_customer
USING DELTA
AS
SELECT
    customer_id,
    customer_name,
    email,
    city,
    address,

    1 AS is_active,

    CURRENT_DATE() AS start_date,

    CAST(NULL AS DATE) AS end_date,

    CURRENT_TIMESTAMP() AS load_timestamp

FROM silver.customers_silver;

In [0]:
%sql
CREATE OR REPLACE TABLE dim_product
USING DELTA
AS
SELECT
    product_id,
    product_name,
    category,
    unit_price,
    silver_load_time AS load_timestamp
FROM silver.products_silver;

In [0]:
%sql
CREATE OR REPLACE TABLE dim_store
USING DELTA
AS
SELECT
    store_id,
    store_name,
    region,
    silver_load_time AS load_timestamp
FROM silver.stores_silver;

In [0]:
%sql
CREATE OR REPLACE TABLE fact_sales
USING DELTA
AS
SELECT
    s.transaction_id,
    s.customer_id,
    c.customer_name,

    s.product_id,
    p.product_name,
    p.category,

    s.store_id,
    st.store_name,
    st.region,

    s.quantity,

    p.unit_price,

    (s.quantity * p.unit_price) AS total_amount,

    s.transaction_date,

    CURRENT_TIMESTAMP() AS fact_load_time

FROM silver.sales_silver s

LEFT JOIN silver.customers_silver c
ON s.customer_id = c.customer_id

LEFT JOIN silver.products_silver p
ON s.product_id = p.product_id

LEFT JOIN silver.stores_silver st
ON s.store_id = st.store_id;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_staging
USING DELTA
AS
SELECT *
FROM silver.customers_silver;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW changed_customers AS

SELECT
    source.*
FROM gold.customer_staging source

JOIN gold.dim_customer target

ON source.customer_id = target.customer_id

WHERE target.is_active = 1

AND (

    target.customer_name <> source.customer_name
    OR target.email <> source.email
    OR target.city <> source.city
    OR target.address <> source.address

);

In [0]:
%sql
UPDATE gold.dim_customer AS target

SET
    target.is_active = 0,
    target.end_date = CURRENT_DATE()

WHERE target.customer_id IN (

    SELECT source.customer_id
    FROM gold.customer_staging source

    WHERE
        target.customer_id = source.customer_id
        AND target.is_active = 1

        AND (

            target.customer_name <> source.customer_name
            OR target.city <> source.city
            OR target.email <> source.email
            OR target.address <> source.address

        )

);

In [0]:
%sql
INSERT INTO gold.dim_customer

SELECT

    source.customer_id,
    source.customer_name,
    source.email,
    source.city,
    source.address,

    1 AS is_active,

    CURRENT_DATE() AS start_date,

    NULL AS end_date,

    CURRENT_TIMESTAMP() AS load_timestamp

FROM gold.customer_staging source

LEFT JOIN gold.dim_customer target

ON source.customer_id = target.customer_id
AND target.is_active = 1

WHERE

    target.customer_id IS NULL

    OR

    (
        target.customer_name <> source.customer_name
        OR target.city <> source.city
        OR target.email <> source.email
        OR target.address <> source.address
    );

In [0]:
%sql
SELECT 'dim_customer' AS table_name, COUNT(*) FROM dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*) FROM dim_product
UNION ALL
SELECT 'dim_store', COUNT(*) FROM dim_store
UNION ALL
SELECT 'fact_sales', COUNT(*) FROM fact_sales;

In [0]:
%sql
SELECT *
FROM fact_sales
WHERE customer_id IS NULL
OR product_id IS NULL
OR store_id IS NULL;


In [0]:
%sql
SELECT *
FROM fact_sales
WHERE total_amount <= 0;

In [0]:
%sql
SELECT
    region,
    SUM(total_amount) AS total_sales
FROM fact_sales
GROUP BY region
ORDER BY total_sales DESC;

In [0]:
%sql
SELECT
    product_name,
    SUM(quantity) AS total_quantity
FROM fact_sales
GROUP BY product_name
ORDER BY total_quantity DESC;

In [0]:
%sql
SELECT
    customer_name,
    SUM(total_amount) AS customer_sales
FROM fact_sales
GROUP BY customer_name
ORDER BY customer_sales DESC;

In [0]:
%sql
SELECT

    customer_id,
    customer_name,
    city,
    is_active

FROM gold.dim_customer

WHERE customer_id IN (1,2,501,502)

ORDER BY customer_id, is_active;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.customer_staging
USING DELTA
AS

SELECT *
FROM silver.customers_silver;

In [0]:
%sql
SELECT

    customer_id,
    customer_name,
    city

FROM gold.customer_staging

WHERE customer_id IN (1,2,501,502);

In [0]:
%sql
SELECT

    customer_id,
    customer_name,
    city,
    is_active,
    start_date,
    end_date

FROM gold.dim_customer

WHERE customer_id IN (1,2,501,502)

ORDER BY customer_id, is_active;